In [1]:
# Import necessary libraries for feature engineering and time-series
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Load the cleaned master dataset from Notebook 1
master_df = pd.read_csv('master_cleaned_sales.csv')

# Ensure 'date' column is in datetime format and sort the dataframe
master_df['date'] = pd.to_datetime(master_df['date'])
master_df = master_df.sort_values(by=['store_id', 'department', 'date']).reset_index(drop=True)

# Display the first few rows and shape to verify
print(f"Dataset Shape: {master_df.shape}")
display(master_df.head())

Dataset Shape: (156000, 18)


,store_id,department,date,weekly_sales,is_holiday,store_type,store_size,region,temperature,fuel_price,markdown_1,markdown_2,markdown_3,markdown_4,markdown_5,cpi,unemployment,season
0,1,1,2022-01-01,119075.96,1,A,213810,North,97.57,4.83,10334.49,5905.86,5261.52,7098.43,876.08,203.52,3.32,Winter
1,1,1,2022-01-08,35525.05,0,A,213810,North,46.03,3.67,1356.75,2486.21,1427.01,983.27,2442.13,196.91,8.62,Winter
2,1,1,2022-01-15,14847.56,0,A,213810,North,25.96,5.46,3861.22,596.15,22.09,2854.11,3180.86,267.48,8.40,Winter
3,1,1,2022-01-22,45254.57,0,A,213810,North,25.92,3.58,579.35,2589.31,2493.19,1158.14,286.01,217.32,5.28,Winter
4,1,1,2022-01-29,15166.69,0,A,213810,North,78.37,4.41,4436.06,1416.64,478.38,2496.36,3423.53,247.35,8.40,Winter


In [2]:
# Create Lag Features and Rolling Features for time-series forecasting
# Sorting to ensure chronological order per store and department
master_df = master_df.sort_values(by=['store_id', 'department', 'date']).reset_index(drop=True)

# 1. Lag Features (e.g., previous week's sales)
master_df['sales_lag_1'] = master_df.groupby(['store_id', 'department'])['weekly_sales'].shift(1)
master_df['sales_lag_4'] = master_df.groupby(['store_id', 'department'])['weekly_sales'].shift(4)

# 2. Rolling Window Features (e.g., 4-week rolling mean of past sales)
master_df['sales_rolling_mean_4'] = master_df.groupby(['store_id', 'department'])['weekly_sales'].transform(lambda x: x.shift(1).rolling(window=4).mean())

# Drop rows with NaN values created due to shifting/rolling
master_df = master_df.dropna().reset_index(drop=True)

print("Lag and Rolling features created successfully!")
print(f"Dataset Shape after feature engineering: {master_df.shape}")
display(master_df[['store_id', 'department', 'date', 'weekly_sales', 'sales_lag_1', 'sales_rolling_mean_4']].head())

Lag and Rolling features created successfully!
Dataset Shape after feature engineering: (152000, 21)


,store_id,department,date,weekly_sales,sales_lag_1,sales_rolling_mean_4
0,1,1,2022-01-29,15166.69,45254.57,53675.7850
1,1,1,2022-02-05,15601.13,15166.69,27698.4675
2,1,1,2022-02-12,52743.22,15601.13,22717.4875
3,1,1,2022-02-19,38995.95,52743.22,32191.4025
4,1,1,2022-02-26,62899.98,38995.95,30626.7475


In [3]:
from statsmodels.tsa.stattools import adfuller

# Function to perform Augmented Dickey-Fuller (ADF) Test
def check_stationarity(series):
    result = adfuller(series.dropna())
    print('ADF Statistic:', result[0])
    print('p-value:', result[1])
    print('Critical Values:')
    for key, value in result[4].items():
        print(f'\t{key}: {value}')
    if result[1] <= 0.05:
        print("Conclusion: Series is Stationary (Reject H0)")
    else:
        print("Conclusion: Series is Non-Stationary (Fail to reject H0)")

# Test stationarity on a sample store/department's weekly sales
sample_series = master_df[(master_df['store_id'] == 1) & (master_df['department'] == 1)]['weekly_sales']
print("Performing ADF Test on Store 1, Dept 1 Weekly Sales:")
check_stationarity(sample_series)

Performing ADF Test on Store 1, Dept 1 Weekly Sales:
ADF Statistic: -3.1227984711247894
p-value: 0.024911362861226128
Critical Values:
	1%: -3.4782936965183815
	5%: -2.882567574015525
	10%: -2.5779830117488745
Conclusion: Series is Stationary (Reject H0)


In [4]:
# 1. Encode categorical features
# Encoding 'season', 'store_type', and 'region'
master_df = pd.get_dummies(master_df, columns=['season', 'store_type', 'region'], drop_first=True)

# 2. Save the engineered dataset
master_engineered_file = 'master_engineered_sales.csv'
master_df.to_csv(master_engineered_file, index=False)

print("Feature Engineering completed successfully!")
print(f"Final Engineered Dataset Shape: {master_df.shape}")
print(f"Dataset successfully saved as '{master_engineered_file}'")
display(master_df.head(2))

Feature Engineering completed successfully!
Final Engineered Dataset Shape: (152000, 26)
Dataset successfully saved as 'master_engineered_sales.csv'


,store_id,department,date,weekly_sales,is_holiday,store_size,temperature,fuel_price,markdown_1,markdown_2,...,sales_lag_4,sales_rolling_mean_4,season_Spring,season_Summer,season_Winter,store_type_B,store_type_C,region_North,region_South,region_West
0,1,1,2022-01-29,15166.69,0,213810,78.37,4.41,4436.06,1416.64,...,119075.96,53675.7850,False,False,True,False,False,True,False,False
1,1,1,2022-02-05,15601.13,0,213810,59.50,4.07,2137.71,76.26,...,35525.05,27698.4675,False,False,True,False,False,True,False,False
